In [1]:
from langgraph.store.memory import InMemoryStore

#1长期记忆-基础API的使用
# put()和get()的使用
store=InMemoryStore()
namespace=("users",)
user_id="user-1"
user_name="小明"
store.put(namespace,user_id,{"name":user_name})
item=store.get(namespace,user_id)
print(item)

Item(namespace=['users'], key='user-1', value={'name': '小明'}, created_at='2026-09-08T13:31:25.076698+00:00', updated_at='2026-09-08T13:31:25.076698+00:00')


In [2]:
# 更新数据
user_name="小红"
store.put(namespace,user_id,{"name":user_name})
item=store.get(namespace,user_id)
print(item)

Item(namespace=['users'], key='user-1', value={'name': '小红'}, created_at='2026-09-08T13:32:47.628986+00:00', updated_at='2026-09-08T13:32:47.628986+00:00')


postgresql的实现

In [5]:
from langgraph.store.postgres import PostgresStore
from getpass import getpass
from psycopg.conninfo import make_conninfo

DB_URL = make_conninfo(
    host="127.0.0.1",
    port=15432,
    user="langchain_user",
    password=getpass("请输入 langchain_user 的数据库密码："),
    dbname="langchain_db",
    sslmode="disable",
    gssencmode="disable",
    connect_timeout=10,
)
with PostgresStore.from_conn_string(DB_URL) as store:
    store.setup()
    print("连接成功，记忆存储表已初始化")
    namespace=("users",)
    user_id="user-1"
    user_name="小明"
    store.put(namespace,user_id,{"name":user_name})
    item=store.get(namespace,user_id)
    print(item)

连接成功，记忆存储表已初始化
Item(namespace=['users'], key='user-1', value={'name': '小明'}, created_at='2026-09-08T21:40:26.887236+08:00', updated_at='2026-09-08T21:40:26.887236+08:00')


In [6]:
with PostgresStore.from_conn_string(DB_URL) as store:
    store.setup()
    print("连接成功，记忆存储表已初始化")
    
    user_name="小红"
    store.put(namespace,user_id,{"name":user_name})
    item=store.get(namespace,user_id)
    print(item)

连接成功，记忆存储表已初始化
Item(namespace=['users'], key='user-1', value={'name': '小红'}, created_at='2026-09-08T21:40:26.887236+08:00', updated_at='2026-09-08T21:43:42.537404+08:00')


2 search()的使用基于内存中数据的存储

In [7]:
store=InMemoryStore()
namespace1=("users","Alice","memories")
key1='preferences'
value1={
    "course":"计算机组成原理",
    "sports":"跑步",
    "food":"紫光园奶皮子酸奶"
}
namespace2=("users","Bob","memorise")
key2='preferences'
value2={
    "course":"数字电路与模拟电路",
    "sports":"跑步",
    "food":"奶皮子糖葫芦"
}
namespace3=("users","Black","memorise")
key3='preferences'
value3={
    "course":"数字电路与模拟电路",
    "sports":"羽毛球",
    "food":"紫光园奶皮子酸奶"
}
store.put(namespace1,key1,value1)
store.put(namespace2,key2,value2)
store.put(namespace3,key3,value3)

In [9]:
for item in store.search(("users","Alice")):
    print(item)

Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-09-08T13:55:23.086677+00:00', updated_at='2026-09-08T13:55:23.086677+00:00', score=None)


2.2按照filter过滤

In [10]:
for item in store.search(("users",),filter={"food":"紫光园奶皮子酸奶"}):
    print(item)

Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-09-08T13:55:23.086677+00:00', updated_at='2026-09-08T13:55:23.086677+00:00', score=None)
Item(namespace=['users', 'Black', 'memorise'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-09-08T13:55:23.086677+00:00', updated_at='2026-09-08T13:55:23.086677+00:00', score=None)


##语言搜索

In [11]:
# 自定义嵌入函数将一段文本转化为多维向量的一个函数
from langgraph.store.memory import InMemoryStore

# 自定义嵌入函数
def embed(text: list[str]) -> list[list[float]]:
    return [[1.0] * 6 for _ in range(len(text))]

index_config = {
    "embed": embed, # 使用嵌入函数或嵌入模型赋值
    "dims": 6,
    "fields": ["$", "course"]
}
store = InMemoryStore(
    index = index_config
)

namespace1 = ("users", "Alice", "memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}

store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)

In [12]:
from pprint import pprint
pprint(store._vectors)

defaultdict(<function InMemoryStore.__init__.<locals>.<lambda> at 0x000001658F7151C0>,
            {('users', 'Alice', 'memories'): defaultdict(<class 'dict'>,
                                                         {'preferences': {'$': [1.0,
                                                                                1.0,
                                                                                1.0,
                                                                                1.0,
                                                                                1.0,
                                                                                1.0],
                                                                          'course': [1.0,
                                                                                     1.0,
                                                                                     1.0,
                                                           